# 07 — Attention and Transformer from scratch in PyTorch

This notebook is part of the NLP implementation pack for AI PMs, founders, and builders. It is designed to be runnable on toy data and easy to adapt to real company data.

## Mental model

Attention lets each token decide which other tokens are relevant.

```text
Attention(Q,K,V) = softmax(QKᵀ / sqrt(dk)) V
```

This notebook implements scaled dot-product attention, multi-head attention, positional encoding, and a tiny causal Transformer language model.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import math
import re
import pandas as pd
import numpy as np
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F

df = pd.read_csv(DATA_DIR / "sample_customer_tickets.csv")
texts = df["text"].tolist()

## Scaled dot-product attention

- Q asks a question
- K offers matchable information
- V carries content forward

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    dk = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(dk)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ V
    return output, weights

# Demo: batch=1, heads=1, seq=3, dim=4
Q = torch.randn(1, 1, 3, 4)
K = torch.randn(1, 1, 3, 4)
V = torch.randn(1, 1, 3, 4)
out, weights = scaled_dot_product_attention(Q, K, V)
print("weights shape:", weights.shape)
print(weights[0,0])

## Multi-head self-attention

Multiple heads let the model learn different relationships in parallel: syntax, entities, position, coreference, sentiment cues, and more.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        def split_heads(t):
            return t.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        q, k, v = map(split_heads, (q, k, v))
        y, weights = scaled_dot_product_attention(q, k, v, mask)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(y), weights

attn = MultiHeadSelfAttention(d_model=32, num_heads=4)
x = torch.randn(2, 5, 32)
y, w = attn(x)
print(y.shape, w.shape)

## Positional encoding

Transformers have no recurrence, so we add position information to token embeddings.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

## Transformer block

A decoder-style block uses masked self-attention so it cannot see future tokens.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=128, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, weights = self.attn(self.ln1(x), mask)
        x = x + self.dropout(attn_out)
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.dropout(ffn_out)
        return x, weights

## Tiny causal language model

This is a toy GPT-like model trained to predict the next token. It demonstrates pretraining mechanics at small scale.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z]+|[.!?]", text.lower())

corpus_tokens = []
for text in texts:
    corpus_tokens.extend(tokenize(text) + ["<eos>"])

vocab = sorted(set(corpus_tokens))
stoi = {w:i for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}
ids = torch.tensor([stoi[w] for w in corpus_tokens], dtype=torch.long)

block_size = 8
X, Y = [], []
for i in range(len(ids) - block_size):
    X.append(ids[i:i+block_size])
    Y.append(ids[i+1:i+block_size+1])
X = torch.stack(X)
Y = torch.stack(Y)
print(X.shape, Y.shape, len(vocab))

In [ ]:
class TinyCausalTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, max_len=64):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, max_len=max_len)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads) for _ in range(num_layers)])
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        x = self.pos(self.token_emb(idx))
        # causal mask: 1 means allowed, 0 means masked
        mask = torch.tril(torch.ones(T, T, device=idx.device)).view(1, 1, T, T)
        last_weights = None
        for block in self.blocks:
            x, last_weights = block(x, mask)
        logits = self.head(self.ln(x))
        return logits, last_weights

lm = TinyCausalTransformerLM(len(vocab))
optimizer = torch.optim.AdamW(lm.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(20):
    logits, _ = lm(X)
    loss = loss_fn(logits.view(-1, len(vocab)), Y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 5 == 0:
        print(epoch+1, float(loss))

In [ ]:
def generate(model, prompt="the app", max_new_tokens=15):
    model.eval()
    toks = [stoi.get(t, 0) for t in tokenize(prompt)]
    idx = torch.tensor(toks, dtype=torch.long).unsqueeze(0)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
        if itos[int(next_id)] == "<eos>":
            break
    return " ".join(itos[int(i)] for i in idx[0])

for prompt in ["the app", "i love", "the model"]:
    print(generate(lm, prompt))

## Product note

You rarely train Transformers from scratch as a startup unless you have massive data and compute. But understanding this notebook helps you reason about:

- why context windows matter
- why generation is autoregressive
- why attention cost grows with sequence length
- what pretraining means mechanically